# Lesson 03 - Agentic Design Patterns

Šioje pamokoje nagrinėsime tris pagrindinius efektyvių AI agentų kūrimo dizaino modelius:

1. **Aiškios agento instrukcijos** — Tikslūs, vaidmenį apibrėžiantys užklausimai, nukreipiantys agento elgesį
2. **Struktūruotas išvestis su Pydantic modeliais** — Užtikrinant agentų grąžinamų duomenų prognozuojamumą ir patikimumą
3. **Vienos atsakomybės agentai** — Kuriant susitelkusius agentus, kurie gerai atlieka vieną užduotį

Kiekvieną modelį pritaikysime **kelionių tikslų rekomendatoriaus** scenarijui, palaipsniui kuriant sistemą, galinčią siūlyti kelionės vietas, tikrinti prieinamumą ir tvarkyti logistiką.


## Diegimas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 1 Šablonas: aiškios agento instrukcijos

Pačiu veiksmingiausiu šablonu taip pat yra paprasčiausias: rašyti aiškias, detalias instrukcijas savo agentui.

Geros instrukcijos apibrėžia:
- **Kas** yra agentas (asmenybė ir tonas)
- **Ką** jis turėtų daryti (žingsnis po žingsnio atsakomybės)
- **Kaip** jis turėtų elgtis (apribojimai ir stilius)

Žemiau sukuriame kelionių konsjeržo agentą su aiškiomis instrukcijomis, kurios formuoja kiekvieną jo pateiktą atsakymą.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Šablonas 2: Struktūruotas išvestis su Pydantic modeliais

Laisvai rašytas tekstas yra naudingas pokalbiui, tačiau tolesnės sistemos reikalauja struktūruotų duomenų. 
Sujungdami **Pydantic modelius** su **įrankio funkcija**, galime:

- Apibrėžti tikslią agento išvesties schemą  
- Automatiškai tikrinti atsakymus  
- Patikimai integruoti agento rezultatus į programos logiką  

Taip pat pristatome įrankį, kuris pateikia tikslų paskirties vietos informaciją, kad agentas grindžia savo rekomendacijas tikrais duomenimis.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Šablonas 3: Vienos Atsakomybės Agentai

Sudėtingos užduotys naudingos, kai darbas paskirstomas keliems specializuotiems agentams, iš kurių kiekvienas atsakingas už vieną sritį:

- **Vietos Ekspertas**, kuris žino apie vietas ir jų prieinamumą
- **Logistikos Planavimo Specialistas**, kuris rūpinasi skrydžiais, viešbučiais ir maršrutais

Tai atitinka programinės įrangos inžinerijos principą *atskyrimo pagal atsakomybę* — kiekvieną agentą lengviau testuoti, prižiūrėti ir tobulinti nepriklausomai.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Santrauka

Šioje pamokoje pritaikėme tris agentų dizaino šablonus kelionių rekomendavimo scenarijui:

| Šablonas | Pagrindinė idėja | Nauda |
|---|---|---|
| **Aiškios instrukcijos** | Iš anksto apibrėžti asmenybę, atsakomybes ir apribojimus | Nuosekli, prekės ženklui atitinkanti agento elgsena |
| **Struktūrizuotas išvestis** | Naudoti Pydantic modelius kaip atsako formatą | Patikrinti, mašinai skaitomi rezultatai |
| **Vienintelė atsakomybė** | Suteikti kiekvienam agentui vieną fokusuotą užduotį | Lengviau testuoti, prižiūrėti ir derinti |

Šie šablonai natūraliai derinasi – galite sujungti aiškias instrukcijas su struktūruota išvestimi vieno agento, turinčio vienintelę atsakomybę, viduje, kad sukurtumėte tvirtas, gamybai paruoštas sistemas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės atsisakymas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Pirminis dokumentas originalo kalba turėtų būti laikomas patikimiausiu šaltiniu. Esant svarbiai informacijai, rekomenduojama pasitelkti profesionalų žmogaus vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingus aiškinimus, kylančius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
